# B-cell reclustering


In [ ]:
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = (adata_qc.obs['leiden_coarse'] == 'B Cells') & (adata_qc.obs['best_rank_type_global'] == 'B_cell')

adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=550)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=35, n_pcs=30, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.4, key_added='leiden_0.4_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.4_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/b_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict={'0':'B_BANK1',
           '1':'B_CD45_high',
           '2':'B_BACH2',
           '3':'B_RPs',
           '4':'B_LDHB_high',
           '5':'B_MZB1',
           '6':'pDC_TCF4',
          }
adata.obs['cell_subtype'] = adata.obs['leiden_0.4_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_b.h5ad')


# Endothelial-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = (adata_qc.obs['leiden_coarse'] == 'Endothelial Cells') & (adata_qc.obs['best_rank_type_global'] == 'Endo')

adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=50, n_pcs=50, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.4, key_added='leiden_0.4_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.4_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/endo_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'Endo_CXCR4_high',
    '1': 'Endo_PLVAP',
    '2': 'Endo_ENO1_high',
    '3': 'Endo_FLT1',
    '4': 'Endo_SPARC',
    '5': 'Endo_CXCL8_high',
    '6': 'Endo_EFNB2',
    '7': 'Endo_EHD3',
    '8': 'Endo_ACKR1',
    '9': 'Endo_CDK1'
}
adata.obs['cell_subtype'] = adata.obs['leiden_0.4_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_endo.h5ad')


# Mast-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = ((adata_qc.obs['leiden_coarse'] == 'Mast Cells') & (adata_qc.obs['best_rank_type_global'] == 'Mast'))

adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=15, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.8, key_added='leiden_0.8_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.8_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/mast_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'Mast_TPSAB1',
    '1': 'Mast_FYN_high',        # Txibao
    '2': 'Mye_CD163',
    '3': 'Mast_SOCS3_high',
    '4': 'Mast_RPs',
    '5': 'Mast_TPSB2',
    '6': 'Mast_MIF_high',
    '7': 'Mast_CD74_high',
    '8': 'Mast_MTs',
    '9': 'Mast_KIT',
    '10': 'Mast_CXCL8_high',
    '11': 'Mast_MKI67'
}

adata.obs['cell_subtype'] = adata.obs['leiden_0.8_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_mast.h5ad')


# Myeloid-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = ((adata_qc.obs['leiden_coarse'] == 'Myeloid Cells') & (adata_qc.obs['best_rank_type_global'] == 'Myeloid'))
adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=50, n_pcs=10, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.4, key_added='leiden_0.4_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.4_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/mye_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'Mye_SPP1_high',
    '1': 'Mye_CASP8_high',
    '2': 'Mye_FCN1',
    '3': 'Mye_IL1B',
    '4': 'Mye_OLR1',
    '5': 'Mye_C1QA',
    '6': 'Mye_TREM2' ,
    '7': 'Mye_FCGR3A'
}
adata.obs['cell_subtype'] = adata.obs['leiden_0.3_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_mye.h5ad')


# NK-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = ((adata_qc.obs['leiden_coarse'] == 'NK Cells') & (adata_qc.obs['best_rank_type_global'] == 'NK'))
adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=10, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.6, key_added='leiden_0.6_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.6_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/nk_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'NK_MKI67',
    '1': 'NK_KLRF1',          # NK
    '2': 'T_GZMK',
    '3': 'NK_HSPA1A',
    '4': 'NK_XCL1',
    '5': 'NK_CD16',
    '6': 'T_GZMH',
    '7': 'T_TRBC1',          # NK-like T cells

}

# adata.obs['cell_subtype'] = adata.obs['leiden_your_res'].map(anno_dict)
adata.obs['cell_subtype'] = adata.obs['leiden_0.6_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_nk.h5ad')


# Stromal-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = (adata_qc.obs['leiden_coarse'] == 'Stromal Cells') & (adata_qc.obs['best_rank_type_global'] == 'Stromal')

adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=20, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.5, key_added='leiden_0.5_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.5_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/s_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'S_VCAN_high',        # Tumor
    '1': 'S_FXYD5_high',      # T/NK
    '2': 'S_FOS',
    '3': 'S_RGS5',
    '4': 'S_ACTA2',
    '5': 'S_PRKG1',
    '6': 'S_DCN',
    '7': 'S_DAB2_high',
}

adata.obs['cell_subtype'] = adata.obs['leiden_0.5_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_s.h5ad')


# T-cell reclustering


In [ ]:
adata_qc = sc.read_h5ad('/home/lirui/kirc/adata_qc.h5ad')


In [ ]:
adata_anno = sc.read_h5ad('../adata_anno_score_genes_rank_re.h5ad')


In [ ]:
celltype_dict = adata_anno.obs['leiden_coarse'].to_dict()


In [ ]:
celltype_rank = adata_anno.obs['best_rank_type_global'].to_dict()


In [ ]:
adata_qc.obs['leiden_coarse'] = adata_qc.obs.index.map(celltype_dict)
adata_qc.obs['best_rank_type_global'] = adata_qc.obs.index.map(celltype_rank)


In [ ]:
mask = ((adata_qc.obs['leiden_coarse'] == 'T Cells') & (adata_qc.obs['best_rank_type_global'] == 'T_cell'))

adata = adata_qc[mask].copy()


In [ ]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata,target_sum=1e4)
rsc.pp.log1p(adata)
rsc.pp.highly_variable_genes(adata,n_top_genes=3000)
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]].copy()
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_MT"])
rsc.pp.scale(adata,max_value=10)
rsc.tl.pca(adata,n_comps=50)


In [ ]:
rsc.pp.harmony_integrate(adata, key="sample",basis="X_pca",adjusted_basis="X_pca_inte",max_iter_harmony=10)


In [ ]:
rsc.pp.neighbors(adata, n_neighbors=30, n_pcs=15, use_rep='X_pca_inte')
rsc.tl.umap(adata, min_dist=0.5, spread=1.0)


In [ ]:
rsc.tl.leiden(adata, resolution=0.8, key_added='leiden_0.8_detailed')


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden_0.8_detailed', method='t-test')


In [ ]:
result = adata.uns['rank_genes_groups']
for group in result['names'].dtype.names:
    group_data = pd.DataFrame({
        'gene': result['names'][group],
        'score': result['scores'][group],
        'logfoldchanges': result['logfoldchanges'][group],
        'pvals': result['pvals'][group],
        'pvals_adj': result['pvals_adj'][group]
    })

    group_data.to_csv(f'./subtype_degs/t_deg_{group}_filtered_ttest.csv', index=False)


In [ ]:
anno_dict = {
    '0': 'T_FOXP3',
    '1': 'T_TOX',
    '2': 'T_MKI67',
    '3': 'T_GNLY',
    '4': 'T_LAG3',
    '5': 'T_HSPA1A',
    '6': 'T_RPs',
    '7': 'T_IL7R',
    '8': 'T_KLRB1',
}

adata.obs['cell_subtype'] = adata.obs['leiden_0.8_detailed'].map(anno_dict)


In [ ]:
adata.write_h5ad('adata_t.h5ad')


# Cell-subtype DEG export


In [ ]:
from pathlib import Path


In [ ]:
base_dir = Path(".")
output_dir = base_dir / "cellsubtype_degs"
output_dir.mkdir(parents=True, exist_ok=True)

adata_dict = {
    "epi": sc.read_h5ad(base_dir / "adata_epi.h5ad"),
    "b": sc.read_h5ad(base_dir / "adata_b.h5ad"),
    "endo": sc.read_h5ad(base_dir / "adata_endo.h5ad"),
    "mast": sc.read_h5ad(base_dir / "adata_mast.h5ad"),
    "mye": sc.read_h5ad(base_dir / "adata_mye.h5ad"),
    "nk": sc.read_h5ad(base_dir / "adata_nk.h5ad"),
    "s": sc.read_h5ad(base_dir / "adata_s.h5ad"),
    "t": sc.read_h5ad(base_dir / "adata_t.h5ad"),
}


In [ ]:
for major_type, adata in adata_dict.items():
    sc.tl.rank_genes_groups(
        adata,
        groupby="cell_subtype",
        method="t-test",
    )
    result = adata.uns["rank_genes_groups"]

    for cell_subtype in result["names"].dtype.names:
        group_data = pd.DataFrame({
            "gene": result["names"][cell_subtype],
            "score": result["scores"][cell_subtype],
            "logfoldchanges": result["logfoldchanges"][cell_subtype],
            "pvals": result["pvals"][cell_subtype],
            "pvals_adj": result["pvals_adj"][cell_subtype],
        })
        group_data.to_csv(
            output_dir / f"{cell_subtype}_degs_{major_type}.csv",
            index=False,
        )


# Final cell-subtype UMAPs


# Manual selected-size UMAPs for cell subtype plots

Each h5ad is plotted in its own cell without a loop. Colors use the global reference subtype color map. Existing outputs are not overwritten.

In [ ]:
from pathlib import Path
import gc

import matplotlib as mpl
import pandas as pd
import scanpy as sc

base_dir = Path(".")
out_dir = base_dir / "umap_cell_subtype_ref_colors_publication_manual_selected_sizes"
out_dir.mkdir(exist_ok=True)
sc.settings.figdir = str(out_dir)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
sc.set_figure_params(figsize=(3.5, 3.5), dpi=300)

ref_color_df = pd.read_csv(base_dir / "umap_cell_subtype_ref_colors_publication" / "reference_cell_subtype_color_map.csv")
ref_color_map = dict(zip(ref_color_df["cell_subtype"], ref_color_df["color"]))
plot_log_rows = []


## adata_b.h5ad: selected size 6.0

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_b.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=6.0,
    show=False,
    save="_adata_b_umap_cell_subtype_manual_size6p0.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=6.0,
    show=False,
    save="_adata_b_umap_cell_subtype_manual_size6p0.svg",
)
plot_log_rows.append({
    "file": "adata_b.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 6.0,
    "pdf": str(out_dir / "umap_adata_b_umap_cell_subtype_manual_size6p0.pdf"),
    "svg": str(out_dir / "umap_adata_b_umap_cell_subtype_manual_size6p0.svg"),
})
del adata
gc.collect()


## adata_endo.h5ad: selected size 1.5

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_endo.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.5,
    show=False,
    save="_adata_endo_umap_cell_subtype_manual_size1p5.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.5,
    show=False,
    save="_adata_endo_umap_cell_subtype_manual_size1p5.svg",
)
plot_log_rows.append({
    "file": "adata_endo.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 1.5,
    "pdf": str(out_dir / "umap_adata_endo_umap_cell_subtype_manual_size1p5.pdf"),
    "svg": str(out_dir / "umap_adata_endo_umap_cell_subtype_manual_size1p5.svg"),
})
del adata
gc.collect()


## adata_epi.h5ad: selected size 0.8

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_epi.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=0.8,
    show=False,
    save="_adata_epi_umap_cell_subtype_manual_size0p8.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=0.8,
    show=False,
    save="_adata_epi_umap_cell_subtype_manual_size0p8.svg",
)
plot_log_rows.append({
    "file": "adata_epi.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 0.8,
    "pdf": str(out_dir / "umap_adata_epi_umap_cell_subtype_manual_size0p8.pdf"),
    "svg": str(out_dir / "umap_adata_epi_umap_cell_subtype_manual_size0p8.svg"),
})
del adata
gc.collect()


## adata_mast.h5ad: selected size 7.0

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_mast.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=7.0,
    show=False,
    save="_adata_mast_umap_cell_subtype_manual_size7p0.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=7.0,
    show=False,
    save="_adata_mast_umap_cell_subtype_manual_size7p0.svg",
)
plot_log_rows.append({
    "file": "adata_mast.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 7.0,
    "pdf": str(out_dir / "umap_adata_mast_umap_cell_subtype_manual_size7p0.pdf"),
    "svg": str(out_dir / "umap_adata_mast_umap_cell_subtype_manual_size7p0.svg"),
})
del adata
gc.collect()


## adata_mye.h5ad: selected size 1.4

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_mye.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.4,
    show=False,
    save="_adata_mye_umap_cell_subtype_manual_size1p4.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.4,
    show=False,
    save="_adata_mye_umap_cell_subtype_manual_size1p4.svg",
)
plot_log_rows.append({
    "file": "adata_mye.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 1.4,
    "pdf": str(out_dir / "umap_adata_mye_umap_cell_subtype_manual_size1p4.pdf"),
    "svg": str(out_dir / "umap_adata_mye_umap_cell_subtype_manual_size1p4.svg"),
})
del adata
gc.collect()


## adata_nk.h5ad: selected size 1.4

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_nk.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.4,
    show=False,
    save="_adata_nk_umap_cell_subtype_manual_size1p4.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.4,
    show=False,
    save="_adata_nk_umap_cell_subtype_manual_size1p4.svg",
)
plot_log_rows.append({
    "file": "adata_nk.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 1.4,
    "pdf": str(out_dir / "umap_adata_nk_umap_cell_subtype_manual_size1p4.pdf"),
    "svg": str(out_dir / "umap_adata_nk_umap_cell_subtype_manual_size1p4.svg"),
})
del adata
gc.collect()


## adata_s.h5ad: selected size 1.8

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_s.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.8,
    show=False,
    save="_adata_s_umap_cell_subtype_manual_size1p8.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.8,
    show=False,
    save="_adata_s_umap_cell_subtype_manual_size1p8.svg",
)
plot_log_rows.append({
    "file": "adata_s.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 1.8,
    "pdf": str(out_dir / "umap_adata_s_umap_cell_subtype_manual_size1p8.pdf"),
    "svg": str(out_dir / "umap_adata_s_umap_cell_subtype_manual_size1p8.svg"),
})
del adata
gc.collect()


## adata_t.h5ad: selected size 1.0

In [ ]:
adata = sc.read_h5ad(base_dir / "adata_t.h5ad")
if hasattr(adata.obs["cell_subtype"], "cat"):
    adata.obs["cell_subtype"] = adata.obs["cell_subtype"].cat.remove_unused_categories()
    present_categories = list(adata.obs["cell_subtype"].cat.categories)
else:
    present_categories = sorted(adata.obs["cell_subtype"].astype(str).unique())
    adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"].astype(str), categories=present_categories)

missing = [x for x in present_categories if x not in ref_color_map]
if missing:
    raise ValueError("Missing reference colors for " + ", ".join(missing))
adata.uns["cell_subtype_colors"] = [ref_color_map[x] for x in present_categories]

sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.0,
    show=False,
    save="_adata_t_umap_cell_subtype_manual_size1p0.pdf",
)
sc.pl.umap(
    adata,
    color="cell_subtype",
    frameon=True,
    size=1.0,
    show=False,
    save="_adata_t_umap_cell_subtype_manual_size1p0.svg",
)
plot_log_rows.append({
    "file": "adata_t.h5ad",
    "n_obs": int(adata.n_obs),
    "n_cell_subtype": int(adata.obs["cell_subtype"].nunique()),
    "selected_point_size": 1.0,
    "pdf": str(out_dir / "umap_adata_t_umap_cell_subtype_manual_size1p0.pdf"),
    "svg": str(out_dir / "umap_adata_t_umap_cell_subtype_manual_size1p0.svg"),
})
del adata
gc.collect()


In [ ]:
pd.DataFrame(plot_log_rows).to_csv(out_dir / "manual_selected_size_plot_log.csv", index=False)
pd.DataFrame(plot_log_rows)